Cell 1: Configuration (Expanded Themes & Multi-Stock Support)

In [1]:
import os
import re
import io
import requests
import zipfile
import pandas as pd
import numpy as np
import logging
from datetime import datetime, timedelta
from pandas_datareader import data as pdr

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- CONFIGURATION ---

# 1. BROAD THEMES: Captures anything remotely business/economy related
# This prevents "too strict" filtering while still ignoring sports/lifestyle.
FINANCIAL_THEME_KEYWORDS = [
    'ECON', 'FINANCE', 'TAX', 'BANK', 'BUDGET', 'TRADE', 'MARKET', 
    'INVEST', 'STOCKS', 'BUSINESS', 'CORPORATE', 'REGULATION', 
    'WB_698_TRADE', 'EPU_POLICY', 'CRISISLEX' 
]

# 2. TARGET STOCKS & ALIASES
# Logic: If any of these aliases appear in the Title (Slug) OR the Entity column, we take it.
STOCK_CONFIG = {
    'AAPL': {
        'aliases': ['apple inc', 'apple computer', 'apple store', 'iphone', 'ipad', 'macbook', 'aapl'],
        'blacklist': ['apple valley', 'little apple', 'recipe', 'bake', 'pie', 'cider', 'yeast', 'orchard']
    },
    'AMZN': {
        'aliases': ['amazon.com', 'amazon inc', 'aws', 'prime video', 'jeff bezos', 'amzn'],
        'blacklist': ['amazon river', 'rainforest', 'fire', 'forest']
    },
    'GOOGL': {
        'aliases': ['alphabet inc', 'google', 'youtube', 'sundar pichai', 'googl', 'goog'],
        'blacklist': []
    },
    'MSFT': {
        'aliases': ['microsoft', 'azure', 'windows', 'satya nadella', 'msft'],
        'blacklist': []
    },
    'TSLA': {
        'aliases': ['tesla', 'elon musk', 'spacex', 'cybertruck', 'tsla'],
        'blacklist': ['nikola tesla']
    }
}

DATA_DIR = './data'
os.makedirs(DATA_DIR, exist_ok=True)


Cell 2: Data Ingestion (GDELT + Stooq for ALL Tickers)

In [2]:

# --- INGESTION FUNCTIONS ---

def download_daily_gkg(date_str):
    """
    Downloads GKG file, returns DataFrame with essential columns.
    Stable 11-column load.
    """
    url = f"http://data.gdeltproject.org/gkg/{date_str}.gkg.csv.zip"
    logger.info(f"Fetching GKG: {url}")
    
    try:
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            logger.warning(f"URL not found: {url}")
            return pd.DataFrame()
            
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                # Load all 11 standard Bronze columns
                bronze_cols = ['DATE', 'NUMARTS', 'COUNTS', 'THEMES', 'LOCATIONS', 
                               'PERSONS', 'ORGANIZATIONS', 'TONE', 'CAMEOEVENTIDS', 
                               'SOURCES', 'SOURCEURLS']
                
                df = pd.read_csv(f, sep='\t', header=None, names=bronze_cols, 
                                 encoding='utf-8', on_bad_lines='skip', dtype=str)
                
                # Keep only what we need for Silver filtering
                return df[['DATE', 'THEMES', 'ORGANIZATIONS', 'SOURCEURLS']]
                
    except Exception as e:
        logger.error(f"Download failed for {date_str}: {e}")
        return pd.DataFrame()

def ingest_stooq_history(tickers, days_back=365):
    """
    Fetches stock history for MULTIPLE tickers.
    """
    start_date = datetime.now() - timedelta(days=days_back)
    all_stocks = []
    
    for ticker in tickers:
        logger.info(f"Fetching Stooq data for {ticker}...")
        try:
            # Stooq symbol format usually 'AAPL.US' but 'AAPL' often works with pdr
            df = pdr.DataReader(ticker, 'stooq', start_date, datetime.now())
            
            if not df.empty:
                df = df.reset_index()
                # Clean up column names
                df.columns = [c.lower() for c in df.columns]
                df['ticker'] = ticker
                # Stooq often returns index as 'Date'
                if 'date' in df.columns:
                    df['date_obj'] = pd.to_datetime(df['date']).dt.date
                
                all_stocks.append(df[['date_obj', 'ticker', 'close', 'volume', 'open', 'high', 'low']])
        except Exception as e:
            logger.error(f"Failed to fetch {ticker}: {e}")
            
    if all_stocks:
        combined = pd.concat(all_stocks, ignore_index=True)
        # Drop duplicates just in case
        return combined.sort_values(['ticker', 'date_obj']).drop_duplicates()
    return pd.DataFrame()



Cell 3: Silver Processing (Corrected Logic for Multiple Stocks)

In [3]:
# --- SILVER LAYER LOGIC (Strict Title Match) ---

def extract_title_from_url(url):
    """
    Extracts a readable title from the URL slug.
    """
    if not isinstance(url, str): return ""
    try:
        # Match the last part of URL, ignore query params (?) or .html
        slug = re.search(r'([^/]+)(?:\.html|\?|$)', url.strip('/'))
        if slug:
            text = slug.group(1)
            # Remove digits (IDs), replace separators with spaces
            text = re.sub(r'\d+', '', text)
            text = text.replace('-', ' ').replace('_', ' ').replace('+', ' ')
            return text.strip().title()
    except:
        pass
    return ""

def process_daily_news(raw_df, stock_config):
    """
    Filters a SINGLE day of GKG data.
    """
    if raw_df.empty: return pd.DataFrame()
    
    subset_df = raw_df.copy()
    
    # 1. Pre-calculate extraction (We rely entirely on this now)
    subset_df['extracted_title'] = subset_df['SOURCEURLS'].apply(extract_title_from_url)
    
    # Drop bad/short titles immediately
    subset_df = subset_df[subset_df['extracted_title'].str.len() > 15]
    
    processed_rows = []
    
    # 2. Loop through EACH Ticker
    for ticker, rules in stock_config.items():
        aliases = rules['aliases']
        blacklist = rules['blacklist']
        
        # Regex for Aliases
        alias_pattern = '|'.join([re.escape(a) for a in aliases])
        
        # STRICT FILTER: The Title/URL ITSELF must contain the alias.
        # We ignore the 'THEMES' and 'ORGANIZATIONS' columns for filtering now.
        mask_title_strict = subset_df['extracted_title'].str.contains(alias_pattern, case=False, na=False)
        
        # Check Blacklist
        if blacklist:
            blk_pattern = '|'.join([re.escape(b) for b in blacklist])
            mask_black = subset_df['SOURCEURLS'].astype(str).str.contains(blk_pattern, case=False, na=False)
            final_mask = mask_title_strict & (~mask_black)
        else:
            final_mask = mask_title_strict
            
        # Apply mask
        stock_news = subset_df[final_mask].copy()
        
        if not stock_news.empty:
            stock_news['ticker'] = ticker
            processed_rows.append(stock_news[['DATE', 'ticker', 'extracted_title']])
            
    if processed_rows:
        return pd.concat(processed_rows, ignore_index=True)
    return pd.DataFrame()

In [4]:
# --- EXECUTION: Date Range Selection ---

def generate_date_range(start_date_str, end_date_str):
    """
    Generates a list of YYYYMMDD strings between start and end (inclusive).
    Format: 'YYYY-MM-DD'
    """
    start = datetime.strptime(start_date_str, '%Y-%m-%d')
    end = datetime.strptime(end_date_str, '%Y-%m-%d')
    
    date_list = []
    curr = start
    while curr <= end:
        date_list.append(curr.strftime('%Y%m%d'))
        curr += timedelta(days=1)
    return date_list

# === PARAMETERS ===
START_DATE = '2025-12-18' # Adjust as needed
END_DATE   = '2025-12-20' # Adjust as needed
# ==================

# 1. Generate Dates
dates = generate_date_range(START_DATE, END_DATE)
logger.info(f"Processing range: {dates[0]} to {dates[-1]} ({len(dates)} days)")

# 2. Run Pipeline
daily_news_list = []

for d in dates:
    raw_gkg = download_daily_gkg(d)
    if not raw_gkg.empty:
        daily_clean = process_daily_news(raw_gkg, STOCK_CONFIG)
        if not daily_clean.empty:
            daily_news_list.append(daily_clean)
            # Optional: Print count to track progress
            print(f"  > {d}: Found {len(daily_clean)} items.")

if daily_news_list:
    silver_news_df = pd.concat(daily_news_list, ignore_index=True)
    
    # Clean up Date (YYYYMMDD... -> YYYY-MM-DD)
    silver_news_df['date'] = pd.to_datetime(silver_news_df['DATE'].astype(str).str[:8], format='%Y%m%d').dt.date
    
    print(f"\nTotal Clean News Items Found: {len(silver_news_df)}")
    print(silver_news_df.head())
else:
    print("\nNo relevant news found with strict filters.")

2025-12-22 09:32:34,544 - INFO - Processing range: 20251218 to 20251220 (3 days)
2025-12-22 09:32:34,544 - INFO - Fetching GKG: http://data.gdeltproject.org/gkg/20251218.gkg.csv.zip
2025-12-22 09:35:42,967 - INFO - Fetching GKG: http://data.gdeltproject.org/gkg/20251219.gkg.csv.zip


  > 20251218: Found 1094 items.


2025-12-22 09:37:20,962 - INFO - Fetching GKG: http://data.gdeltproject.org/gkg/20251220.gkg.csv.zip


  > 20251219: Found 787 items.
  > 20251220: Found 518 items.

Total Clean News Items Found: 2399
       DATE ticker                                    extracted_title  \
0  20251218   AAPL  Kyivstar Enables Starlink Direct To Mobile Tes...   
1  20251218   AAPL           Apple Opens Iphone Alternative App .Html   
2  20251218   AAPL  Apple Discusses Iphone Chip Assembly With Indi...   
3  20251218   AAPL  Best Free Attraction In Usa According To Tripa...   
4  20251218   AAPL            .Childhood Books Gave Us Escapism Ipads   

         date  
0  2025-12-18  
1  2025-12-18  
2  2025-12-18  
3  2025-12-18  
4  2025-12-18  


Cell 4: NLP Input Generation

In [5]:

# --- OUTPUT 1: NLP INPUT ---
# Requirement: Just Date, Entity, Title. 
# This file goes to your ABSA model.

if 'silver_news_df' in locals() and not silver_news_df.empty:
    nlp_output_df = silver_news_df[['date', 'ticker', 'extracted_title']].copy()
    
    # Rename columns to be crystal clear for the model
    nlp_output_df.columns = ['date', 'entity', 'title']
    
    # Save
    outfile = f'{DATA_DIR}/nlp_input.csv'
    nlp_output_df.to_csv(outfile, index=False)
    print(f"✓ NLP Input saved to {outfile}")
    print(nlp_output_df.head())


✓ NLP Input saved to ./data/nlp_input.csv
         date entity                                              title
0  2025-12-18   AAPL  Kyivstar Enables Starlink Direct To Mobile Tes...
1  2025-12-18   AAPL           Apple Opens Iphone Alternative App .Html
2  2025-12-18   AAPL  Apple Discusses Iphone Chip Assembly With Indi...
3  2025-12-18   AAPL  Best Free Attraction In Usa According To Tripa...
4  2025-12-18   AAPL            .Childhood Books Gave Us Escapism Ipads


Cell 5: Machine Learning Training Data Prep (Placeholder)

In [6]:

# --- OUTPUT 2: ML TRAINING DATA PREP ---
# This step assumes you have run the NLP model and have a file called 'nlp_output_scored.csv'
# containing: date, entity, title, sentiment_score.
# Since we don't have that yet, we will MOCK it to show the merge logic with Stooq.

def create_training_set(stock_df, sentiment_df):
    """
    Merges Daily Stock Price with Daily Aggregated Sentiment
    """
    # Aggregate sentiment: Mean score + Count (Volume) per day/ticker
    daily_sentiment = sentiment_df.groupby(['date', 'entity']).agg({
        'sentiment_score': 'mean',
        'title': 'count' # Using title count as 'news_volume'
    }).reset_index().rename(columns={
        'date': 'date_obj', 
        'entity': 'ticker',
        'sentiment_score': 'avg_sentiment', 
        'title': 'news_count'
    })
    
    # Merge
    # We left join stock_df (master) with sentiment (features)
    # Ensure dates match types (both dates or both timestamps)
    merged = pd.merge(stock_df, daily_sentiment, on=['date_obj', 'ticker'], how='left')
    
    # Fill NaN news days with 0 (No news = Neutral 0 score, 0 volume)
    merged['avg_sentiment'] = merged['avg_sentiment'].fillna(0)
    merged['news_count'] = merged['news_count'].fillna(0)
    
    # Create Target: Next Day's Close (Simple Shift)
    # Logic: Today's News + Today's Close -> Predict Tomorrow's Close
    merged['target_next_close'] = merged.groupby('ticker')['close'].shift(-1)
    
    return merged.dropna()

# --- MOCK EXECUTION ---
if 'nlp_output_df' in locals() and not stock_history_df.empty:
    # 1. Mock the NLP scoring (Replace this with actual model loading later)
    mock_scored = nlp_output_df.copy()
    mock_scored['sentiment_score'] = np.random.uniform(-1, 1, size=len(mock_scored)) # Mock scores
    
    # 2. Run logic
    training_data = create_training_set(stock_history_df, mock_scored)
    
    outfile_ml = f'{DATA_DIR}/ml_training_data.csv'
    training_data.to_csv(outfile_ml, index=False)
    
    print(f"✓ ML Training Data saved to {outfile_ml}")
    print(training_data[['date_obj', 'ticker', 'close', 'avg_sentiment', 'news_count', 'target_next_close']].tail())

NameError: name 'stock_history_df' is not defined